# 🧠 Artificial Neural Network (ANN)
**Tabular Classification with PyTorch**
---

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, auc

print(f'PyTorch version : {torch.__version__}')
print('Libraries loaded ✅')

## 2. Load & Explore Dataset
> Using the **Breast Cancer Wisconsin** dataset — 569 samples, 30 features, binary classification (Malignant / Benign).

In [ ]:
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

print(f'Shape   : {df.shape}')
print(f'Classes : {df["target"].value_counts().to_dict()}')
df.head()

## 3. Data Preprocessing
> Neural networks require scaled input features for stable and fast convergence. We use `StandardScaler` to normalize features to zero mean and unit variance.

In [ ]:
X = df.drop('target', axis=1).values.astype(np.float32)
y = df['target'].values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape : {X_test.shape}')

## 4. Build the ANN Model

In [ ]:
class ANNClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dims=[64, 32], dropout=0.3):
        super(ANNClassifier, self).__init__()
        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))  # Stabilizes training
            layers.append(nn.ReLU())              # Non-linearity
            layers.append(nn.Dropout(dropout))    # Prevents overfitting
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, 1))
        layers.append(nn.Sigmoid())               # Output probability
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

model = ANNClassifier(input_dim=X_train.shape[1], hidden_dims=[64, 32], dropout=0.3)
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f'\\nTotal Parameters: {total_params:,}')

## 5. Train the Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train).unsqueeze(1))
test_dataset = TensorDataset(torch.tensor(X_test), torch.tensor(y_test).unsqueeze(1))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 100
train_losses, val_accs = [], []

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        outputs = model(bx)
        loss = criterion(outputs, by)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * bx.size(0)
    
    train_losses.append(epoch_loss / len(train_dataset))
    
    # Validation
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for bx, by in test_loader:
            bx, by = bx.to(device), by.to(device)
            outputs = model(bx)
            preds = (outputs >= 0.5).float()
            val_correct += (preds == by).sum().item()
            val_total += bx.size(0)
    
    val_accs.append(val_correct / val_total)
    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1}/{epochs} | Loss: {train_losses[-1]:.4f} | Val Acc: {val_accs[-1]:.4f}')

print('Training complete!')

## 6. Evaluate on Test Set

In [ ]:
model.eval()
all_preds, all_targets, all_probs = [], [], []
with torch.no_grad():
    for bx, by in test_loader:
        bx = bx.to(device)
        outputs = model(bx)
        probs = outputs.cpu().numpy().flatten()
        preds = (outputs >= 0.5).float().cpu().numpy().flatten()
        all_preds.extend(preds)
        all_targets.extend(by.numpy().flatten())
        all_probs.extend(probs)

acc = accuracy_score(all_targets, all_preds)
prec = precision_score(all_targets, all_preds, zero_division=0)
rec = recall_score(all_targets, all_preds, zero_division=0)
f1 = f1_score(all_targets, all_preds, zero_division=0)

print('='*50)
print('          ANN Test Set Results')
print('='*50)
print(f'  Accuracy  : {acc:.4f}')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print('='*50)

## 7. Confusion Matrix & ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(all_targets, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Malignant (0)', 'Benign (1)'],
            yticklabels=['Malignant (0)', 'Benign (1)'],
            linewidths=1, linecolor='white')
axes[0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')

# ROC Curve
fpr, tpr, thresholds = roc_curve(all_targets, all_probs)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='#818cf8', lw=2.5, label=f'ANN (AUC = {roc_auc:.4f})')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#818cf8')
axes[1].plot([0,1],[0,1],'k--', lw=1.5, label='Random Classifier')
axes[1].set_title('ROC Curve', fontsize=14, fontweight='bold')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

plt.tight_layout(); plt.show()

## 8. Save Model & Scaler

In [ ]:
import os, joblib
os.makedirs('../models', exist_ok=True)
torch.save(model.state_dict(), '../models/ann_model.pth')
joblib.dump(scaler, '../models/scaler.pkl')
print('Model saved  → models/ann_model.pth')
print('Scaler saved → models/scaler.pkl')

## 9. Key Takeaways
> - **Scaling is Critical**: Neural networks are highly sensitive to unscaled input features. Always use `StandardScaler` or `MinMaxScaler`.
> - **Batch Normalization**: Placed after Linear layers, it stabilizes and accelerates training by normalizing layer inputs.
> - **Dropout**: Randomly zeroes out neurons during training, preventing the network from relying too heavily on specific features (overfitting).
> - **PyTorch Flexibility**: Building custom architectures with dynamic layer sizes is straightforward using `nn.Sequential` or custom `nn.Module` classes.